In [50]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, cast, current_date, current_timestamp, to_date, row_number, date_format, year, sum, desc, spark_partition_id
from pyspark.sql.window import Window

In [5]:
# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [8]:
spark= (SparkSession
        .builder
        .master("local[*]")
        .appName('spark transformation')
        .getOrCreate()
       )

In [10]:
spark

In [9]:
emp_df = spark.createDataFrame(data=emp_data , schema=emp_schema)

In [11]:
emp_df.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [25]:
emp_df_date = ( 
                emp_df.withColumn('hire_date', to_date(col('hire_date'), format('yyyy-MM-dd')))
               .withColumn('hire_year', year(col('hire_date')))
                .withColumn('salary', col('salary').cast('int'))
              )

In [26]:
emp_df_date.show()

+-----------+-------------+-------------+---+------+------+----------+---------+
|employee_id|department_id|         name|age|gender|salary| hire_date|hire_year|
+-----------+-------------+-------------+---+------+------+----------+---------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|     2015|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|     2016|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|     2014|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|     2017|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|     2013|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|     2018|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|     2012|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|     2019|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|     2016|
|        010|          104| 

In [27]:
emp_df_date.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- hire_year: integer (nullable = true)



In [36]:
agg_df = emp_df_date.groupBy("department_id") \
    .agg(sum("salary").alias("total_salary")) \
    .orderBy("total_salary", ascending=False)


In [38]:
agg_df.show()

+-------------+------------+
|department_id|total_salary|
+-------------+------------+
|          103|      232000|
|          102|      207000|
|          101|      165000|
|          104|      162000|
|          106|      138000|
|          105|      111000|
|          107|       95000|
+-------------+------------+



In [39]:
dept_data = [
    ["101", "Sales", "NYC", "US", "1000000"],
    ["102", "Marketing", "LA", "US", "900000"],
    ["103", "Finance", "London", "UK", "1200000"],
    ["104", "Engineering", "Beijing", "China", "1500000"],
    ["105", "Human Resources", "Tokyo", "Japan", "800000"],
    ["106", "Research and Development", "Perth", "Australia", "1100000"],
    ["107", "Customer Service", "Sydney", "Australia", "950000"]
]

dept_schema = "department_id string, department_name string, city string, country string, budget string"

In [40]:
dept = spark.createDataFrame(data=dept_data, schema=dept_schema)

In [41]:
dept.show()

+-------------+--------------------+-------+---------+-------+
|department_id|     department_name|   city|  country| budget|
+-------------+--------------------+-------+---------+-------+
|          101|               Sales|    NYC|       US|1000000|
|          102|           Marketing|     LA|       US| 900000|
|          103|             Finance| London|       UK|1200000|
|          104|         Engineering|Beijing|    China|1500000|
|          105|     Human Resources|  Tokyo|    Japan| 800000|
|          106|Research and Deve...|  Perth|Australia|1100000|
|          107|    Customer Service| Sydney|Australia| 950000|
+-------------+--------------------+-------+---------+-------+



In [42]:
emp_df.rdd.getNumPartitions()

8

In [43]:
dept.rdd.getNumPartitions()

8

In [47]:
emp_partition = emp_df.repartition(4, 'department_id')

In [48]:
emp_partition.rdd.getNumPartitions()

4

In [51]:
emp_1 = emp_partition.withColumn('Number of partition', spark_partition_id())

In [52]:
emp_1.show()

+-----------+-------------+-------------+---+------+------+----------+-------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|Number of partition|
+-----------+-------------+-------------+---+------+------+----------+-------------------+
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|                  0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|                  0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|                  0|
|        014|          107|    Emily Lee| 26|Female| 46000|2019-01-01|                  0|
|        016|          107|  Kelly Zhang| 30|Female| 49000|2018-04-01|                  0|
|        020|          102|    Grace Kim| 32|Female| 53000|2018-11-01|                  0|
|        012|          105|   Susan Chen| 31|Female| 54000|2017-02-15|                  1|
|        017|          105|  George Wang| 34|  Male| 57000|2016-03-15|                  1|

In [53]:
df_joined = emp_df.join(dept, how='inner', on=emp_df.department_id == dept.department_id)

In [54]:
df_joined.show()

+-----------+-------------+-------------+---+------+------+----------+-------------+--------------------+-------+---------+-------+
|employee_id|department_id|         name|age|gender|salary| hire_date|department_id|     department_name|   city|  country| budget|
+-----------+-------------+-------------+---+------+------+----------+-------------+--------------------+-------+---------+-------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|          101|               Sales|    NYC|       US|1000000|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|          101|               Sales|    NYC|       US|1000000|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|          101|               Sales|    NYC|       US|1000000|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|          102|           Marketing|     LA|       US| 900000|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|      

In [59]:
df_joinedl = emp_df.alias('e').join(dept.alias('d'), how='left_outer', on=emp_df.department_id == dept.department_id)

In [60]:
df_joinedl.select('e.name', 'd.department_id', 'e.salary').show()

+-------------+-------------+------+
|         name|department_id|salary|
+-------------+-------------+------+
|     John Doe|          101| 50000|
|   Jane Smith|          101| 45000|
|    Bob Brown|          102| 55000|
|    Alice Lee|          102| 48000|
|    Jack Chan|          103| 60000|
|    Jill Wong|          103| 52000|
|James Johnson|          101| 70000|
|     Lisa Lee|          104| 47000|
|     Kate Kim|          102| 51000|
|      Tom Tan|          103| 58000|
|   David Park|          104| 65000|
|   Susan Chen|          105| 54000|
|    Emily Lee|          107| 46000|
|    Brian Kim|          106| 75000|
|  Kelly Zhang|          107| 49000|
|  Michael Lee|          106| 63000|
|    Nancy Liu|          104| 50000|
|    Grace Kim|          102| 53000|
|  Steven Chen|          103| 62000|
|  George Wang|          105| 57000|
+-------------+-------------+------+



In [64]:
df_final = emp_df.join(dept, how='left_outer', 
                       on=(emp_df.department_id == dept.department_id) 
                       & (emp_df.department_id == 101) | (emp_df.department_id == 102))

In [65]:
df_final.show()

+-----------+-------------+-------------+---+------+------+----------+-------------+--------------------+-------+---------+-------+
|employee_id|department_id|         name|age|gender|salary| hire_date|department_id|     department_name|   city|  country| budget|
+-----------+-------------+-------------+---+------+------+----------+-------------+--------------------+-------+---------+-------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|          101|               Sales|    NYC|       US|1000000|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|          101|               Sales|    NYC|       US|1000000|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|          101|               Sales|    NYC|       US|1000000|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|          102|           Marketing|     LA|       US| 900000|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|      